# Why transform data to get a straight line?

For the fundamental frequency of a stretched string,

$$
f=\frac{1}{2L}\sqrt{\frac{T}{\mu}}
$$

If $T$ and $\mu$ are fixed, then

$$
f \propto \frac{1}{L}
$$

But imagine we do **not** know this yet. We only have measured pairs of string length $L$ and frequency $f$.

The aim is to see why finding a transformation that produces a straight line is useful.

In [1]:
%pip install ipywidgets  # needed by the in-browser (Pyodide) kernel
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

## 1. Generate some ukulele-like data

The defaults are illustrative rather than tied to one exact string:

- $T \approx 35\,\mathrm{N}$
- $\mu \approx 1.3\times10^{-3}\,\mathrm{kg\,m^{-1}}$
- lengths around a typical ukulele scale length

A small random measurement error is added.

In [3]:
T = widgets.FloatSlider(value=35,min=10,max=80,step=1,description='T / N')
mu = widgets.FloatLogSlider(value=1.3e-3,base=10,min=-4,max=-2,step=0.05,description='mu')
noise = widgets.FloatSlider(value=0.8,min=0,max=5,step=0.1,description='noise / Hz')
seed = widgets.IntSlider(value=4,min=1,max=100,description='seed')
display(widgets.VBox([T,mu,noise,seed]))

In [4]:
def make_data(T, mu, noise, seed):
    rng = np.random.default_rng(seed)
    L = np.array([0.26,0.285,0.31,0.335,0.36,0.385,0.41])
    f_exact = (1/(2*L))*np.sqrt(T/mu)
    f = f_exact + rng.normal(0,noise,len(L))
    return L,f_exact,f

## 2. Cycle through transformations

> **Can we transform the independent variable so the points lie on a straight line?**

A straight line has the form

$$
y=mx+c.
$$

So if a transformed graph is linear, the relationship becomes much more specific.

Keep frequency $f$ on the vertical axis and try:

- $L$
- $\sqrt{L}$
- $L^2$
- $\ln L$
- $\frac{1}{L}$
- $\frac{1}{\sqrt{L}}$

One of these should produce a straight line.

In [6]:
choice = widgets.ToggleButtons(
    options=[('L','L'),('sqrt(L)','sqrtL'),('L²','L2'),('ln(L)','lnL'),('1/L','invL'),('1/sqrt(L)','invsqrtL')],
    description='x-axis:'
)
display(choice)

ToggleButtons(description='x-axis:', options=(('L', 'L'), ('sqrt(L)', 'sqrtL'), ('L²', 'L2'), ('ln(L)', 'lnL')…

In [7]:
choice = widgets.ToggleButtons(
    options=[
        ('L', 'L'),
        ('√L', 'sqrtL'),
        ('L²', 'L2'),
        ('ln L', 'lnL'),
        ('1/L', 'invL'),
        ('1/√L', 'invsqrtL')
    ],
    value='L',
    description='x-axis:'
)


def transform(L, choice):
    return {
        'L': (L, 'L / m'),
        'sqrtL': (np.sqrt(L), '√L'),
        'L2': (L**2, 'L² / m²'),
        'lnL': (np.log(L), 'ln(L)'),
        'invL': (1 / L, '1/L / m⁻¹'),
        'invsqrtL': (1 / np.sqrt(L), '1/√L')
    }[choice]


def transformed_plot(choice, T, mu, noise, seed):

    L, _, f = make_data(T, mu, noise, seed)

    x, xlabel = transform(L, choice)

    # Best-fit straight line
    m, c = np.polyfit(x, f, 1)
    fit = m * x + c

    # R²
    ss_res = np.sum((f - fit)**2)
    ss_tot = np.sum((f - f.mean())**2)
    r2 = 1 - ss_res / ss_tot

    order = np.argsort(x)

    fig, ax = plt.subplots(figsize=(9, 5))

    ax.scatter(
        x,
        f,
        s=65,
        label='Measurements'
    )

    ax.plot(
        x[order],
        fit[order],
        label='Best-fit straight line'
    )

    ax.set_xlabel(xlabel)
    ax.set_ylabel('Frequency f / Hz')
    ax.set_title(f'Straight-line test   R² = {r2:.5f}')

    ax.grid(alpha=0.3)
    ax.legend()

    plt.show()

    print(f'Best-fit line: f = {m:.3g}x + {c:.3g}')
    print(f'R² = {r2:.5f}')

    if choice == 'invL':
        print()
        print('For x = 1/L:')
        print('f = m(1/L) + c')
        print('Ideal model: c = 0 and m = ½√(T/μ)')


ui = widgets.interactive(
    transformed_plot,
    choice=choice,
    T=T,
    mu=mu,
    noise=noise,
    seed=seed
)

display(ui)

interactive(children=(ToggleButtons(description='x-axis:', options=(('L', 'L'), ('√L', 'sqrtL'), ('L²', 'L2'),…

### 3. Why \(1/L\) gives a straight line

For the string model,

$$
f=\frac{1}{2L}\sqrt{\frac{T}{\mu}}
$$

Rearrange it as

$$
f=
\left(\frac12\sqrt{\frac{T}{\mu}}\right)
\left(\frac1L\right).
$$

Compare with

$$
y=mx+c.
$$

So if we plot $$f$$ against $$1/L$$,

$$
y=f,\qquad x=\frac1L,\qquad
m=\frac12\sqrt{\frac{T}{\mu}},\qquad c=0.
$$

That is why the transformed graph is expected to be straight.

### Suggested teaching sequence
1. Show three measured points.
2. Show several different curves that can pass through them.
3. Ask: **which equation is actually right?**
4. Add more measurements.
5. Cycle through the possible x-axis transformations.
6. Find that $$f$$ against $$1/L$$ is linear.
7. Only then connect it to $$y=mx+c$$.
